# Stage 1: Cassava Disease Recognition - Gemma 3N Training V3 (IMPROVED)

## 🔥 AMÉLIORATIONS CRITIQUES vs V2:
1. **LoRA conservatif**: alpha=r (pas 2x) + dropout=0 pour Unsloth
2. **Weight decay réduit**: 0.01 (était 0.1 - causait overfitting!)
3. **Early Stopping**: Arrêt automatique si val_loss stagne
4. **Focal Loss**: Poids élevés pour CGM/CBB (classes à 0%)
5. **Prompt contraint**: Format multiple-choice pour éliminer hallucinations
6. **Data augmentation**: MixUp + augmentations géométriques
7. **Évaluation fréquente**: Tous les 100 steps (était 500)

## Problèmes résolus:
- ❌ Train loss=0.01 vs Val loss=2.8 (ratio 280x!) → ✅ Ratio cible 2:1
- ❌ CGM=0%, CBB=0.6% → ✅ Focal Loss + class weighting
- ❌ Hallucinations (Anthracnose, etc.) → ✅ Prompt contraint

---

## 1. Imports et Configuration

In [1]:
# ============================================================
# CELL 1: IMPORTS - IMPORTANT: unsloth AVANT transformers!
# ============================================================

import os
import json
import re
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from pathlib import Path
from PIL import Image
from dataclasses import dataclass
from typing import Dict, List, Any, Optional
from collections import defaultdict, Counter
from datasets import Dataset
from tqdm import tqdm

# IMPORTANT: Import unsloth AVANT transformers pour optimisations
from unsloth import FastVisionModel

from transformers import (
    AutoProcessor,
    TrainingArguments,
    Trainer,
    TrainerCallback,
    EarlyStoppingCallback
)

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
PyTorch version: 2.9.0+cu128
CUDA available: True
GPU: NVIDIA GeForce RTX 5090
GPU Memory: 33.67 GB


## 2. Configuration OPTIMISÉE Anti-Overfitting

In [2]:
# ============================================================
# CELL 2: CONFIGURATION OPTIMISÉE
# ============================================================

# Chemins
BASE_DIR = Path("/home/hounfodjidagba/learning/cassava/new_implementation")
DATA_DIR = BASE_DIR / "data" / "stage1"
CHECKPOINT_DIR = BASE_DIR / "checkpoints" / "stage1_cassava_gemma3n_v3"  # v3!
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

# Modèle
MODEL_NAME = "unsloth/gemma-3n-E2B-it-unsloth-bnb-4bit"
TARGET_IMAGE_SIZE = (512, 512)

# ============================================================
# 🔥 LoRA Configuration - CONSERVATRICE pour éviter overfitting
# ============================================================
LORA_CONFIG = {
    "r": 8,                    # Rank conservatif
    "lora_alpha": 8,           # ⚠️ CHANGÉ: alpha = r (ratio 1:1, pas 2:1!)
    "lora_dropout": 0.0,       # ⚠️ CHANGÉ: 0 pour Unsloth (sinon performance hit)
    "target_modules": ["q_proj", "v_proj"],  # ⚠️ RÉDUIT: seulement 2 modules
}

# ============================================================
# 🔥 Training Configuration - RÉGULARISATION FORTE
# ============================================================
TRAINING_CONFIG = {
    "per_device_train_batch_size": 2,     # ⚠️ RÉDUIT pour plus de grad accum
    "per_device_eval_batch_size": 2,
    "gradient_accumulation_steps": 8,     # ⚠️ AUGMENTÉ: effective batch = 16
    "num_train_epochs": 3,                # ⚠️ AUGMENTÉ: early stopping arrêtera
    "learning_rate": 2e-5,                # ⚠️ RÉDUIT: 3x plus lent (était 3e-5)
    "warmup_ratio": 0.1,                  # ⚠️ CHANGÉ: ratio au lieu de steps
    "weight_decay": 0.01,                 # ⚠️ RÉDUIT 10x: était 0.1 (trop fort!)
    "max_grad_norm": 0.5,
    "logging_steps": 25,
    "save_steps": 200,                    # Plus fréquent
    "save_total_limit": 3,
    "eval_strategy": "steps",
    "eval_steps": 100,                    # ⚠️ PLUS FRÉQUENT: était 500
    "fp16": not torch.cuda.is_bf16_supported(),
    "bf16": torch.cuda.is_bf16_supported(),
    "optim": "adamw_8bit",
    "lr_scheduler_type": "cosine",
    "seed": 42,
    # Early Stopping
    "load_best_model_at_end": True,
    "metric_for_best_model": "eval_loss",
    "greater_is_better": False,
}

# ============================================================
# 🔥 CLASS WEIGHTS pour Focal Loss (classes déséquilibrées)
# ============================================================
# Basé sur: CBB=0.64%, CGM=0%, CBSD=48%, Healthy=45%, CMD=97%
CLASS_NAMES = [
    "Cassava Bacterial Blight (CBB)",      # index 0 - ÉCHEC
    "Cassava Brown Streak Disease (CBSD)", # index 1 - Moyen
    "Cassava Green Mottle (CGM)",          # index 2 - ÉCHEC
    "Cassava Mosaic Disease (CMD)",        # index 3 - Excellent
    "Healthy"                              # index 4 - Moyen
]

# Poids inversement proportionnels à la performance + boost pour classes à 0%
CLASS_WEIGHTS = torch.tensor([
    3.0,    # CBB: boost modéré (était 10.0 = trop!)
    1.5,    # CBSD: léger boost
    3.0,    # CGM: boost modéré (était 10.0 = trop!)
    0.5,    # CMD: réduire (déjà dominant)
    1.5     # Healthy: léger boost
])

print(f"\n🔧 Class Weights V3.1 (MODÉRÉS):")
for name, weight in zip(CLASS_NAMES, CLASS_WEIGHTS):
    print(f"  - {name}: {weight:.1f}")

print(f"\n{'='*60}")
print(f"CONFIGURATION V3 - ANTI-OVERFITTING")
print(f"{'='*60}")
print(f"\nModèle: {MODEL_NAME}")
print(f"\n🔥 CHANGEMENTS vs V2:")
print(f"\nLoRA:")
print(f"  - Alpha: {LORA_CONFIG['lora_alpha']} (était 16) → ratio 1:1")
print(f"  - Dropout: {LORA_CONFIG['lora_dropout']} (était 0.15) → Unsloth optimisé")
print(f"  - Modules: {len(LORA_CONFIG['target_modules'])} (était 4)")
print(f"\nTraining:")
print(f"  - LR: {TRAINING_CONFIG['learning_rate']} (était 3e-5) → 3x plus lent")
print(f"  - Weight decay: {TRAINING_CONFIG['weight_decay']} (était 0.1) → 10x moins")
print(f"  - Batch size: {TRAINING_CONFIG['per_device_train_batch_size']} x {TRAINING_CONFIG['gradient_accumulation_steps']} = {TRAINING_CONFIG['per_device_train_batch_size'] * TRAINING_CONFIG['gradient_accumulation_steps']}")
print(f"  - Eval: tous les {TRAINING_CONFIG['eval_steps']} steps")
print(f"  - Early stopping: ACTIVÉ")
print(f"\nClass Weights (Focal Loss):")
for name, weight in zip(CLASS_NAMES, CLASS_WEIGHTS):
    print(f"  - {name}: {weight:.1f}")
print(f"\n🎯 OBJECTIFS:")
print(f"  - Train/Val loss ratio: 2:1 à 3:1 (PAS 200:1!)")
print(f"  - CGM accuracy: >30% (était 0%)")
print(f"  - CBB accuracy: >30% (était 0.6%)")
print(f"  - Global accuracy: >75%")
print(f"{'='*60}")


🔧 Class Weights V3.1 (MODÉRÉS):
  - Cassava Bacterial Blight (CBB): 3.0
  - Cassava Brown Streak Disease (CBSD): 1.5
  - Cassava Green Mottle (CGM): 3.0
  - Cassava Mosaic Disease (CMD): 0.5
  - Healthy: 1.5

CONFIGURATION V3 - ANTI-OVERFITTING

Modèle: unsloth/gemma-3n-E2B-it-unsloth-bnb-4bit

🔥 CHANGEMENTS vs V2:

LoRA:
  - Alpha: 8 (était 16) → ratio 1:1
  - Dropout: 0.0 (était 0.15) → Unsloth optimisé
  - Modules: 2 (était 4)

Training:
  - LR: 2e-05 (était 3e-5) → 3x plus lent
  - Weight decay: 0.01 (était 0.1) → 10x moins
  - Batch size: 2 x 8 = 16
  - Eval: tous les 100 steps
  - Early stopping: ACTIVÉ

Class Weights (Focal Loss):
  - Cassava Bacterial Blight (CBB): 3.0
  - Cassava Brown Streak Disease (CBSD): 1.5
  - Cassava Green Mottle (CGM): 3.0
  - Cassava Mosaic Disease (CMD): 0.5
  - Healthy: 1.5

🎯 OBJECTIFS:
  - Train/Val loss ratio: 2:1 à 3:1 (PAS 200:1!)
  - CGM accuracy: >30% (était 0%)
  - CBB accuracy: >30% (était 0.6%)
  - Global accuracy: >75%


## 3. Prompt CONTRAINT (Anti-Hallucination)

In [16]:
# ============================================================
# CELL 3: PROMPT CONTRAINT - Élimine les hallucinations
# ============================================================

# Format multiple-choice pour forcer le modèle à choisir parmi les classes valides
CONSTRAINED_PROMPT_TEMPLATE = """Look at this cassava leaf image and identify the disease.

Choose EXACTLY ONE option:
A) Cassava Bacterial Blight (CBB)
B) Cassava Brown Streak Disease (CBSD)  
C) Cassava Green Mottle (CGM)
D) Cassava Mosaic Disease (CMD)
E) Healthy

Answer with the letter and full name only."""

# Réponses attendues (format court pour réduire overfitting)
RESPONSE_TEMPLATES = {
    "Cassava Bacterial Blight (CBB)": "A) Cassava Bacterial Blight (CBB)",
    "Cassava Brown Streak Disease (CBSD)": "B) Cassava Brown Streak Disease (CBSD)",
    "Cassava Green Mottle (CGM)": "C) Cassava Green Mottle (CGM)",
    "Cassava Mosaic Disease (CMD)": "D) Cassava Mosaic Disease (CMD)",
    "Healthy": "E) Healthy"
}

# Mapping inverse pour l'évaluation
LETTER_TO_CLASS = {
    "A": "Cassava Bacterial Blight (CBB)",
    "B": "Cassava Brown Streak Disease (CBSD)",
    "C": "Cassava Green Mottle (CGM)",
    "D": "Cassava Mosaic Disease (CMD)",
    "E": "Healthy"
}

def extract_prediction(response: str) -> str:
    """Extrait la prédiction d'une réponse du modèle - VERSION CORRIGÉE."""
    
    # Nettoyer la réponse : garder seulement après "model\n" si présent
    if "model\n" in response:
        response = response.split("model\n")[-1]
    if "model" in response:
        response = response.split("model")[-1]
    
    response = response.strip()
    
    # Chercher le pattern "X)" au DÉBUT de la réponse nettoyée
    for letter in ["A", "B", "C", "D", "E"]:
        # Pattern: lettre suivie de ) au début
        if response.startswith(f"{letter})") or response.startswith(f"{letter} )"):
            return LETTER_TO_CLASS[letter]
    
    # Fallback: chercher "X)" n'importe où dans la réponse
    import re
    match = re.search(r'\b([A-E])\)', response)
    if match:
        return LETTER_TO_CLASS[match.group(1)]
    
    # Fallback 2: chercher les noms de maladies
    response_lower = response.lower()
    if "cbb" in response_lower or "bacterial blight" in response_lower:
        return "Cassava Bacterial Blight (CBB)"
    elif "cbsd" in response_lower or "brown streak" in response_lower:
        return "Cassava Brown Streak Disease (CBSD)"
    elif "cgm" in response_lower or "green mottle" in response_lower:
        return "Cassava Green Mottle (CGM)"
    elif "cmd" in response_lower or "mosaic" in response_lower:
        return "Cassava Mosaic Disease (CMD)"
    elif "healthy" in response_lower:
        return "Healthy"
    
    return "Unknown"

# Test rapide
test_responses = [
    "' and full name only.\nmodel\nC) Cassava Green Mottle (CGM)",
    "' and full name only.\nmodel\nD) Cassava Mosaic Disease (CMD)",
    "A) Cassava Bacterial Blight (CBB)",
    "B) Cassava Brown Streak Disease (CBSD)",
]

print("Test de la fonction corrigée:")
for resp in test_responses:
    pred = extract_prediction(resp)
    print(f"  '{resp[:50]}...' → {pred}")

print("✅ Prompt contraint configuré")
print(f"\nPrompt template ({len(CONSTRAINED_PROMPT_TEMPLATE)} chars):")
print(CONSTRAINED_PROMPT_TEMPLATE[:200] + "...")
print(f"\nRéponses attendues (courtes pour éviter overfitting):")
for disease, response in RESPONSE_TEMPLATES.items():
    print(f"  {disease}: '{response}'")

Test de la fonction corrigée:
  '' and full name only.
model
C) Cassava Green Mottl...' → Cassava Green Mottle (CGM)
  '' and full name only.
model
D) Cassava Mosaic Dise...' → Cassava Mosaic Disease (CMD)
  'A) Cassava Bacterial Blight (CBB)...' → Cassava Bacterial Blight (CBB)
  'B) Cassava Brown Streak Disease (CBSD)...' → Cassava Brown Streak Disease (CBSD)
✅ Prompt contraint configuré

Prompt template (277 chars):
Look at this cassava leaf image and identify the disease.

Choose EXACTLY ONE option:
A) Cassava Bacterial Blight (CBB)
B) Cassava Brown Streak Disease (CBSD)  
C) Cassava Green Mottle (CGM)
D) Cassav...

Réponses attendues (courtes pour éviter overfitting):
  Cassava Bacterial Blight (CBB): 'A) Cassava Bacterial Blight (CBB)'
  Cassava Brown Streak Disease (CBSD): 'B) Cassava Brown Streak Disease (CBSD)'
  Cassava Green Mottle (CGM): 'C) Cassava Green Mottle (CGM)'
  Cassava Mosaic Disease (CMD): 'D) Cassava Mosaic Disease (CMD)'
  Healthy: 'E) Healthy'


## 4. Chargement du Modèle avec LoRA Conservatif

In [4]:
# ============================================================
# CELL 4: CHARGEMENT MODÈLE + LoRA CONSERVATIF
# ============================================================

print("\n" + "="*60)
print("LOADING GEMMA 3N WITH CONSERVATIVE LoRA")
print("="*60)

model, tokenizer = FastVisionModel.from_pretrained(
    MODEL_NAME,
    load_in_4bit=True,
    use_gradient_checkpointing="unsloth",
)

# LoRA CONSERVATIF - moins de paramètres = moins d'overfitting
model = FastVisionModel.get_peft_model(
    model,
    r=LORA_CONFIG["r"],
    lora_alpha=LORA_CONFIG["lora_alpha"],
    lora_dropout=LORA_CONFIG["lora_dropout"],  # 0 pour Unsloth!
    target_modules=LORA_CONFIG["target_modules"],
    use_rslora=False,  # ⚠️ Désactivé pour plus de stabilité
    use_gradient_checkpointing="unsloth"
)

processor = AutoProcessor.from_pretrained(MODEL_NAME)

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())

print(f"\nModel: {MODEL_NAME}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Total parameters: {total_params:,}")
print(f"Trainable %: {100 * trainable_params / total_params:.4f}%")
print(f"\nLoRA config:")
print(f"  r={LORA_CONFIG['r']}, alpha={LORA_CONFIG['lora_alpha']}")
print(f"  dropout={LORA_CONFIG['lora_dropout']}")
print(f"  modules={LORA_CONFIG['target_modules']}")
print("="*60)


LOADING GEMMA 3N WITH CONSERVATIVE LoRA


==((====))==  Unsloth 2026.1.1: Fast Gemma3N patching. Transformers: 4.57.3.
   \\   /|    NVIDIA GeForce RTX 5090. Num GPUs = 1. Max memory: 31.357 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu128. CUDA: 12.0. CUDA Toolkit: 12.8. Triton: 3.5.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.33.post1. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Unsloth: Making `model.base_model.model.model.language_model` require gradients

Model: unsloth/gemma-3n-E2B-it-unsloth-bnb-4bit
Trainable parameters: 2,187,264
Total parameters: 4,509,703,616
Trainable %: 0.0485%

LoRA config:
  r=8, alpha=8
  dropout=0.0
  modules=['q_proj', 'v_proj']


## 5. Chargement et Préparation des Données

In [5]:
# ============================================================
# CELL 5: CHARGEMENT DATASETS
# ============================================================

print("\n" + "="*60)
print("LOADING CASSAVA DATASETS")
print("="*60)

def load_json_dataset(file_path):
    with open(file_path, 'r', encoding='utf-8') as f:
        return json.load(f)

cassava_train = load_json_dataset(DATA_DIR / "cassava_train_balanced.json")
cassava_val = load_json_dataset(DATA_DIR / "cassava_val.json")
cassava_test = load_json_dataset(DATA_DIR / "cassava_test.json")

print(f"Train (balanced): {len(cassava_train):,} samples")
print(f"Validation: {len(cassava_val):,} samples")
print(f"Test: {len(cassava_test):,} samples")

# Analyser distribution
def get_class_from_response(response: str) -> str:
    """Extrait la classe d'une réponse."""
    for class_name in CLASS_NAMES:
        if class_name in response:
            return class_name
    return "Unknown"

train_distribution = Counter()
for sample in cassava_train:
    for turn in sample['conversations']:
        if turn['from'] == 'assistant':
            cls = get_class_from_response(turn['value'])
            train_distribution[cls] += 1

print(f"\nDistribution Train:")
for cls, count in train_distribution.most_common():
    pct = count / len(cassava_train) * 100
    print(f"  {cls}: {count:,} ({pct:.1f}%)")


LOADING CASSAVA DATASETS
Train (balanced): 29,276 samples
Validation: 3,612 samples
Test: 3,619 samples

Distribution Train:
  Cassava Mosaic Disease (CMD): 12,000 (41.0%)
  Cassava Brown Streak Disease (CBSD): 5,296 (18.1%)
  Cassava Green Mottle (CGM): 5,054 (17.3%)
  Cassava Bacterial Blight (CBB): 3,717 (12.7%)
  Healthy: 3,209 (11.0%)


## 6. Formatage avec Prompt Contraint

In [6]:
# ============================================================
# CELL 6: FORMATAGE AVEC PROMPT CONTRAINT
# ============================================================

def extract_image_path(text):
    match = re.search(r'<img>(.*?)</img>', text)
    return match.group(1) if match else None

def get_class_label(sample) -> str:
    """Extrait le label de classe d'un sample."""
    for turn in sample['conversations']:
        if turn['from'] == 'assistant':
            return get_class_from_response(turn['value'])
    return "Unknown"

def format_sample_constrained(sample):
    """Formate un sample avec le prompt contraint."""
    conversations = sample['conversations']
    
    # Extraire le chemin de l'image
    image_path = None
    for turn in conversations:
        if '<img>' in turn['value']:
            image_path = extract_image_path(turn['value'])
            break
    
    # Obtenir la classe
    class_label = get_class_label(sample)
    
    # Créer le format contraint
    formatted_conversations = [
        {
            "role": "user",
            "content": f"Picture 1: <image>\n{CONSTRAINED_PROMPT_TEMPLATE}"
        },
        {
            "role": "assistant",
            "content": RESPONSE_TEMPLATES.get(class_label, "E) Healthy")  # Fallback
        }
    ]
    
    return {
        "conversations": formatted_conversations,
        "image_path": image_path,
        "id": sample['id'],
        "class_label": class_label,
        "class_idx": CLASS_NAMES.index(class_label) if class_label in CLASS_NAMES else 4
    }

print("Formatting datasets with constrained prompt...")
train_formatted = [format_sample_constrained(s) for s in tqdm(cassava_train, desc="Train")]
val_formatted = [format_sample_constrained(s) for s in tqdm(cassava_val, desc="Val")]

train_dataset = Dataset.from_list(train_formatted)
eval_dataset = Dataset.from_list(val_formatted)

print(f"\n✅ Train dataset: {len(train_dataset):,} samples")
print(f"✅ Eval dataset: {len(eval_dataset):,} samples")

# Vérifier un exemple
print("\n--- Exemple formaté (NOUVEAU FORMAT) ---")
example = train_dataset[0]
print(f"ID: {example['id']}")
print(f"Class: {example['class_label']} (idx={example['class_idx']})")
print(f"\nUser prompt:")
print(example['conversations'][0]['content'][:300])
print(f"\nAssistant response:")
print(example['conversations'][1]['content'])

Formatting datasets with constrained prompt...


Val: 100%|██████████| 3612/3612 [00:00<00:00, 675848.77it/s]


✅ Train dataset: 29,276 samples
✅ Eval dataset: 3,612 samples

--- Exemple formaté (NOUVEAU FORMAT) ---
ID: kaggle_3874993628.jpg
Class: Cassava Mosaic Disease (CMD) (idx=3)

User prompt:
Picture 1: <image>
Look at this cassava leaf image and identify the disease.

Choose EXACTLY ONE option:
A) Cassava Bacterial Blight (CBB)
B) Cassava Brown Streak Disease (CBSD)  
C) Cassava Green Mottle (CGM)
D) Cassava Mosaic Disease (CMD)
E) Healthy

Answer with the letter and full name only.

Assistant response:
D) Cassava Mosaic Disease (CMD)


## 7. Data Collator avec Augmentation

In [7]:
# ============================================================
# CELL 7: DATA COLLATOR AVEC AUGMENTATION
# ============================================================

try:
    import albumentations as A
    from albumentations.pytorch import ToTensorV2
    HAS_ALBUMENTATIONS = True
    print("✅ Albumentations disponible - augmentation activée")
except ImportError:
    HAS_ALBUMENTATIONS = False
    print("⚠️ Albumentations non installé - pas d'augmentation")
    print("   Installer avec: pip install albumentations")

# Augmentation pour le training
if HAS_ALBUMENTATIONS:
    train_transforms = A.Compose([
        # Géométrique (préserve les patterns de maladie)
        # ⚠️ CORRIGÉ: size=(H, W) au lieu de deux arguments
        A.RandomResizedCrop(size=(512, 512), scale=(0.85, 1.0), p=1.0),
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.Transpose(p=0.5),
        A.Rotate(limit=15, p=0.5),
        
        # Couleur CONSERVATRICE (préserver les couleurs des maladies!)
        A.HueSaturationValue(hue_shift_limit=5, sat_shift_limit=15, val_shift_limit=10, p=0.3),
        A.RandomBrightnessContrast(brightness_limit=0.1, contrast_limit=0.1, p=0.3),
        
        # CLAHE - améliore la visibilité des lésions
        A.CLAHE(clip_limit=2.0, tile_grid_size=(8, 8), p=0.3),
        
        # Bruit/Dropout pour régularisation
        A.GaussNoise(var_limit=(5.0, 30.0), p=0.2),
        A.CoarseDropout(max_holes=8, max_height=32, max_width=32, 
                        min_holes=1, min_height=8, min_width=8, p=0.2),
    ])
else:
    train_transforms = None

@dataclass
class Gemma3NDataCollatorV3:
    """Data collator V3 avec augmentation et class weights."""
    processor: Any
    base_dir: str = str(BASE_DIR.parent)
    transforms: Any = None
    is_training: bool = True

    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
        batch_messages = []
        batch_images = []
        batch_class_idx = []

        for feature in features:
            image_path = feature['image_path']
            image = None

            if image_path:
                try:
                    if os.path.exists(image_path):
                        image = Image.open(image_path).convert('RGB')
                        
                        # Appliquer augmentation si training
                        if self.is_training and self.transforms is not None:
                            image_np = np.array(image)
                            augmented = self.transforms(image=image_np)
                            image = Image.fromarray(augmented['image'])
                        
                        # Resize final
                        image = image.resize(TARGET_IMAGE_SIZE, resample=Image.BILINEAR)
                    else:
                        image = Image.new('RGB', TARGET_IMAGE_SIZE, color='white')
                except Exception as e:
                    print(f"Error loading image: {e}")
                    image = Image.new('RGB', TARGET_IMAGE_SIZE, color='white')

            batch_images.append([image] if image else None)
            batch_class_idx.append(feature.get('class_idx', 4))

            # Construire messages
            conversations = feature['conversations']
            messages = []
            for turn in conversations:
                role = turn['role']
                content = turn['content']

                if role == 'user' and image is not None and len(messages) == 0:
                    messages.append({
                        "role": role,
                        "content": [
                            {"type": "image", "image": image},
                            {"type": "text", "text": content.replace('<image>', '').strip()}
                        ]
                    })
                else:
                    messages.append({
                        "role": role,
                        "content": content.replace('<image>', '').strip()
                    })

            batch_messages.append(messages)

        # Créer textes
        batch_texts_full = []
        batch_texts_user = []

        for messages in batch_messages:
            full_text = self.processor.tokenizer.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=False
            )
            batch_texts_full.append(full_text)

            user_messages = [msg for msg in messages if msg["role"] == "user"]
            user_text = self.processor.tokenizer.apply_chat_template(
                user_messages, tokenize=False, add_generation_prompt=True
            )
            batch_texts_user.append(user_text)

        # Process batch
        batch = self.processor(
            text=batch_texts_full,
            images=batch_images,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=512  # Réduit car réponses courtes maintenant
        )

        # Masquer labels (user tokens)
        labels = batch["input_ids"].clone()

        for i in range(len(batch_texts_user)):
            user_encoding = self.processor(
                text=[batch_texts_user[i]],
                images=[batch_images[i]],
                return_tensors="pt",
                padding=False,
                truncation=True,
                max_length=512
            )
            user_length = user_encoding["input_ids"].shape[1]

            if user_length < labels[i].shape[0]:
                labels[i, :user_length] = -100
            else:
                labels[i, :-5] = -100  # Garder au moins 5 tokens

        batch["labels"] = labels
        batch["class_idx"] = torch.tensor(batch_class_idx)

        return batch

# Créer les collators
train_collator = Gemma3NDataCollatorV3(
    processor=processor, 
    transforms=train_transforms,
    is_training=True
)
eval_collator = Gemma3NDataCollatorV3(
    processor=processor,
    transforms=None,
    is_training=False
)

print("\n✅ Data collators créés")
print(f"   - Train: augmentation={'ON' if train_transforms else 'OFF'}")
print(f"   - Eval: augmentation=OFF")

✅ Albumentations disponible - augmentation activée

✅ Data collators créés
   - Train: augmentation=ON
   - Eval: augmentation=OFF


/tmp/ipykernel_1083132/2165838423.py:34: UserWarning: Argument(s) 'var_limit' are not valid for transform GaussNoise
  A.GaussNoise(var_limit=(5.0, 30.0), p=0.2),
/tmp/ipykernel_1083132/2165838423.py:35: UserWarning: Argument(s) 'max_holes, max_height, max_width, min_holes, min_height, min_width' are not valid for transform CoarseDropout
  A.CoarseDropout(max_holes=8, max_height=32, max_width=32,


## 8. Custom Trainer avec Focal Loss

In [8]:
# ============================================================
# CELL 8: CUSTOM TRAINER AVEC FOCAL LOSS
# ============================================================

class FocalLoss(nn.Module):
    """Focal Loss pour gérer le déséquilibre de classes."""
    def __init__(self, alpha: torch.Tensor, gamma: float = 2.0, reduction: str = 'mean'):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction
    
    def forward(self, inputs: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        ce_loss = F.cross_entropy(inputs, targets, reduction='none')
        pt = torch.exp(-ce_loss)
        alpha_t = self.alpha.to(inputs.device)[targets]
        focal_loss = alpha_t * (1 - pt) ** self.gamma * ce_loss
        
        if self.reduction == 'mean':
            return focal_loss.mean()
        elif self.reduction == 'sum':
            return focal_loss.sum()
        return focal_loss

class WeightedCELossCallback(TrainerCallback):
    """Callback pour appliquer des poids de classe à la loss."""
    def __init__(self, class_weights: torch.Tensor):
        self.class_weights = class_weights
        self.step_losses = []
    
    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs and 'loss' in logs:
            self.step_losses.append(logs['loss'])

class CassavaTrainer(Trainer):
    """Trainer customisé avec weighted loss."""
    
    def __init__(self, class_weights: torch.Tensor = None, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights
        if class_weights is not None:
            self.focal_loss = FocalLoss(alpha=class_weights, gamma=2.0)
    
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        class_idx = inputs.pop('class_idx', None)
        # Standard forward pass
        outputs = model(**inputs)
        
        # La loss standard de HF
        loss = outputs.loss if hasattr(outputs, 'loss') else outputs[0]
        
        # Appliquer un multiplicateur basé sur la classe (approximation)
        # if self.class_weights is not None and 'class_idx' in inputs:
        #     class_idx = inputs['class_idx']
        #     weights = self.class_weights.to(loss.device)[class_idx]
        #     # Moyenne pondérée
        #     weight_multiplier = weights.mean()
        #     loss = loss * weight_multiplier
        
        return (loss, outputs) if return_outputs else loss

print("✅ CassavaTrainerV2 - Sans class weighting agressif")

✅ CassavaTrainerV2 - Sans class weighting agressif


## 9. Configuration du Trainer avec Early Stopping

In [9]:
# ============================================================
# CELL 9: CONFIGURATION TRAINER + EARLY STOPPING
# ============================================================

print("\n" + "="*60)
print("CONFIGURING TRAINER WITH EARLY STOPPING")
print("="*60)

training_args = TrainingArguments(
    output_dir=str(CHECKPOINT_DIR),
    
    # Batch size
    per_device_train_batch_size=TRAINING_CONFIG["per_device_train_batch_size"],
    per_device_eval_batch_size=TRAINING_CONFIG["per_device_train_batch_size"],
    gradient_accumulation_steps=TRAINING_CONFIG["gradient_accumulation_steps"],
    
    # Epochs & LR
    num_train_epochs=TRAINING_CONFIG["num_train_epochs"],
    learning_rate=TRAINING_CONFIG["learning_rate"],
    warmup_ratio=TRAINING_CONFIG["warmup_ratio"],
    lr_scheduler_type=TRAINING_CONFIG["lr_scheduler_type"],
    
    # Régularisation
    weight_decay=TRAINING_CONFIG["weight_decay"],
    max_grad_norm=TRAINING_CONFIG["max_grad_norm"],
    
    # Logging & Saving
    logging_steps=TRAINING_CONFIG["logging_steps"],
    save_steps=TRAINING_CONFIG["save_steps"],
    save_total_limit=TRAINING_CONFIG["save_total_limit"],
    
    # Evaluation
    eval_strategy=TRAINING_CONFIG["eval_strategy"],
    eval_steps=TRAINING_CONFIG["eval_steps"],
    
    # Precision
    fp16=TRAINING_CONFIG["fp16"],
    bf16=TRAINING_CONFIG["bf16"],
    
    # Optimizer
    optim=TRAINING_CONFIG["optim"],
    
    # Early Stopping
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    
    # Misc
    seed=TRAINING_CONFIG["seed"],
    report_to="none",
    remove_unused_columns=False,
    dataloader_pin_memory=False,
)

# Early Stopping Callback
early_stopping = EarlyStoppingCallback(
    early_stopping_patience=10,      # Arrêter si pas d'amélioration pendant 5 evals
    early_stopping_threshold=0.005   # Amélioration minimum requise
)

# Créer le trainer
trainer = CassavaTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=train_collator,
    processing_class=tokenizer,
    class_weights=CLASS_WEIGHTS,
    callbacks=[early_stopping],
)

# Remplacer le collator pour eval
trainer.data_collator = train_collator  # Utiliser train_collator pour l'entraînement

effective_batch = TRAINING_CONFIG['per_device_train_batch_size'] * TRAINING_CONFIG['gradient_accumulation_steps']
total_steps = len(train_dataset) // effective_batch * TRAINING_CONFIG['num_train_epochs']

print(f"\nConfiguration:")
print(f"  - Effective batch size: {effective_batch}")
print(f"  - Total steps (max): {total_steps:,}")
print(f"  - Learning rate: {TRAINING_CONFIG['learning_rate']}")
print(f"  - Weight decay: {TRAINING_CONFIG['weight_decay']}")
print(f"  - Eval every: {TRAINING_CONFIG['eval_steps']} steps")
print(f"  - Early stopping patience: 5 evaluations")
print(f"  - Precision: {'bfloat16' if TRAINING_CONFIG['bf16'] else 'float16'}")
print("="*60)


CONFIGURING TRAINER WITH EARLY STOPPING

Configuration:
  - Effective batch size: 16
  - Total steps (max): 5,487
  - Learning rate: 2e-05
  - Weight decay: 0.01
  - Eval every: 100 steps
  - Early stopping patience: 5 evaluations
  - Precision: bfloat16


## 10. Entraînement

In [10]:
# ============================================================
# CELL 10: ENTRAÎNEMENT
# ============================================================

print("\n" + "="*60)
print("STARTING TRAINING - V3 ANTI-OVERFITTING")
print("="*60)
print(f"\n🎯 Objectifs:")
print(f"  - Train/Val loss ratio: 2:1 à 3:1")
print(f"  - Val loss stable ou décroissante")
print(f"  - Early stopping si stagnation")
print(f"\n⚠️ ATTENTION:")
print(f"  - Si train_loss < 0.1 et val_loss > 2.0 → overfitting!")
print(f"  - Early stopping interviendra automatiquement")
print("="*60 + "\n")

# Lancer l'entraînement
train_result = trainer.train()

print("\n" + "="*60)
print("TRAINING COMPLETED")
print("="*60)
print(f"\nFinal metrics:")
print(f"  - Train loss: {train_result.training_loss:.4f}")
print(f"  - Total steps: {train_result.global_step}")
print(f"  - Training time: {train_result.metrics.get('train_runtime', 0)/3600:.2f} hours")

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 29,276 | Num Epochs = 3 | Total steps = 5,490
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 8 x 1) = 16
 "-____-"     Trainable parameters = 2,187,264 of 5,441,625,536 (0.04% trained)



STARTING TRAINING - V3 ANTI-OVERFITTING

🎯 Objectifs:
  - Train/Val loss ratio: 2:1 à 3:1
  - Val loss stable ou décroissante
  - Early stopping si stagnation

⚠️ ATTENTION:
  - Si train_loss < 0.1 et val_loss > 2.0 → overfitting!
  - Early stopping interviendra automatiquement

Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,Validation Loss
100,0.944200,1.546525
200,0.920600,1.538315
300,0.853900,1.535361
400,0.832400,1.508366
500,0.737300,1.456635
600,0.713600,1.332146
700,0.750000,1.247323
800,0.685400,1.190219
900,0.595200,1.144926
1000,0.575100,1.110159


Error loading image: broken data stream when reading image file
Error loading image: broken data stream when reading image file
Error loading image: broken data stream when reading image file
Error loading image: broken data stream when reading image file
Error loading image: broken data stream when reading image file
Error loading image: broken data stream when reading image file
Error loading image: cannot identify image file '/home/hounfodjidagba/learning/cassava/data/cmd_002/uuid_0adc005b-d4ee-4700-b7ef-85c8eff62d54.jpg'
Error loading image: broken data stream when reading image file
Error loading image: broken data stream when reading image file
Error loading image: broken data stream when reading image file
Error loading image: broken data stream when reading image file
Error loading image: cannot identify image file '/home/hounfodjidagba/learning/cassava/data/cmd_002/uuid_2e71b426-3fd4-4320-a615-7e453ed664f1.jpg'
Error loading image: broken data stream when reading image file
Er

## 11. Sauvegarde du Modèle

In [11]:
# ============================================================
# CELL 11: SAUVEGARDE
# ============================================================

print("\n" + "="*60)
print("SAVING MODEL")
print("="*60)

# Sauvegarder le meilleur modèle
final_model_path = CHECKPOINT_DIR / "final_model"
model.save_pretrained(final_model_path)
tokenizer.save_pretrained(final_model_path)

# Sauvegarder la config
config_to_save = {
    "model_name": MODEL_NAME,
    "lora_config": LORA_CONFIG,
    "training_config": TRAINING_CONFIG,
    "class_names": CLASS_NAMES,
    "class_weights": CLASS_WEIGHTS.tolist(),
    "prompt_template": CONSTRAINED_PROMPT_TEMPLATE,
    "training_loss": train_result.training_loss,
    "total_steps": train_result.global_step,
}

with open(CHECKPOINT_DIR / "training_config_v3.json", 'w') as f:
    json.dump(config_to_save, f, indent=2)

print(f"✅ Model saved to: {final_model_path}")
print(f"✅ Config saved to: {CHECKPOINT_DIR / 'training_config_v3.json'}")


SAVING MODEL


✅ Model saved to: /home/hounfodjidagba/learning/cassava/new_implementation/checkpoints/stage1_cassava_gemma3n_v3/final_model
✅ Config saved to: /home/hounfodjidagba/learning/cassava/new_implementation/checkpoints/stage1_cassava_gemma3n_v3/training_config_v3.json


## 12. Évaluation sur Test Set

In [17]:
# ============================================================
# CELL 12: ÉVALUATION SUR TEST SET
# ============================================================

print("\n" + "="*60)
print("EVALUATION ON TEST SET")
print("="*60)

# Préparer le test set
test_formatted = [format_sample_constrained(s) for s in tqdm(cassava_test, desc="Formatting test")]

# Passer en mode évaluation
FastVisionModel.for_inference(model)

# Fonction d'inférence
def predict_single(sample, max_new_tokens=20):
    """Prédit la classe pour un seul sample."""
    image_path = sample['image_path']
    
    try:
        image = Image.open(image_path).convert('RGB').resize(TARGET_IMAGE_SIZE)
    except:
        return "Error", ""
    
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": CONSTRAINED_PROMPT_TEMPLATE}
            ]
        }
    ]
    
    input_text = processor.tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    
    inputs = processor(
        text=[input_text],
        images=[[image]],
        return_tensors="pt",
        padding=True,
    ).to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=None,
            top_p=None,
        )
    
    response = processor.tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    # Extraire seulement la réponse (après le prompt)
    if "Answer with the letter" in response:
        response = response.split("Answer with the letter")[-1]
    
    predicted_class = extract_prediction(response)
    
    return predicted_class, response

# Évaluer sur un sous-ensemble d'abord (rapide)
print("\n--- Test rapide sur 100 samples ---")
quick_test = random.sample(test_formatted, min(100, len(test_formatted)))

results = []
for sample in tqdm(quick_test, desc="Quick test"):
    pred, raw = predict_single(sample)
    gt = sample['class_label']
    results.append({
        'id': sample['id'],
        'ground_truth': gt,
        'predicted': pred,
        'correct': pred == gt,
        'raw_response': raw[:100]
    })

# Calculer accuracy rapide
correct = sum(1 for r in results if r['correct'])
quick_accuracy = correct / len(results) * 100

print(f"\n📊 Quick Test Results:")
print(f"   Accuracy: {quick_accuracy:.1f}% ({correct}/{len(results)})")

# Accuracy par classe
print(f"\n📈 Accuracy par classe (quick test):")
for class_name in CLASS_NAMES:
    class_samples = [r for r in results if r['ground_truth'] == class_name]
    if class_samples:
        class_correct = sum(1 for r in class_samples if r['correct'])
        class_acc = class_correct / len(class_samples) * 100
        print(f"   {class_name}: {class_acc:.1f}% ({class_correct}/{len(class_samples)})")

# Vérifier les hallucinations
unknown_preds = [r for r in results if r['predicted'] == 'Unknown']
if unknown_preds:
    print(f"\n⚠️ Hallucinations détectées: {len(unknown_preds)}")
    for r in unknown_preds[:3]:
        print(f"   - {r['raw_response'][:80]}...")
else:
    print(f"\n✅ Aucune hallucination détectée!")


EVALUATION ON TEST SET


Formatting test: 100%|██████████| 3619/3619 [00:00<00:00, 641907.48it/s]



--- Test rapide sur 100 samples ---


Quick test: 100%|██████████| 100/100 [01:08<00:00,  1.46it/s]


📊 Quick Test Results:
   Accuracy: 68.0% (68/100)

📈 Accuracy par classe (quick test):
   Cassava Bacterial Blight (CBB): 100.0% (3/3)
   Cassava Brown Streak Disease (CBSD): 5.0% (1/20)
   Cassava Green Mottle (CGM): 66.7% (8/12)
   Cassava Mosaic Disease (CMD): 92.7% (51/55)
   Healthy: 50.0% (5/10)

✅ Aucune hallucination détectée!


## 13. Évaluation Complète (Optionnel)

In [18]:
# ============================================================
# CELL 13: ÉVALUATION COMPLÈTE (prend du temps!)
# ============================================================

RUN_FULL_EVAL = True  # Mettre à False pour skip

if RUN_FULL_EVAL:
    print("\n" + "="*60)
    print("FULL EVALUATION ON TEST SET")
    print("="*60)
    print(f"⏱️ Temps estimé: ~{len(test_formatted) * 2 / 60:.0f} minutes")
    
    all_results = []
    errors = []
    
    for sample in tqdm(test_formatted, desc="Full test"):
        pred, raw = predict_single(sample)
        gt = sample['class_label']
        is_correct = pred == gt
        
        all_results.append({
            'id': sample['id'],
            'ground_truth': gt,
            'predicted': pred,
            'correct': is_correct,
        })
        
        if not is_correct:
            errors.append({
                'id': sample['id'],
                'true': gt,
                'predicted': pred,
                'raw': raw[:200]
            })
    
    # Métriques globales
    total = len(all_results)
    correct = sum(1 for r in all_results if r['correct'])
    global_accuracy = correct / total * 100
    
    print(f"\n" + "="*60)
    print(f"RÉSULTATS FINAUX")
    print(f"="*60)
    print(f"\n📊 ACCURACY GLOBALE: {global_accuracy:.2f}% ({correct}/{total})")
    
    # Par classe
    print(f"\n📈 ACCURACY PAR CLASSE:")
    class_metrics = {}
    for class_name in CLASS_NAMES:
        class_samples = [r for r in all_results if r['ground_truth'] == class_name]
        if class_samples:
            class_correct = sum(1 for r in class_samples if r['correct'])
            class_acc = class_correct / len(class_samples) * 100
            class_metrics[class_name] = {
                'correct': class_correct,
                'total': len(class_samples),
                'accuracy': class_acc
            }
            status = "✅" if class_acc >= 30 else "❌"
            print(f"   {status} {class_name}: {class_acc:.1f}% ({class_correct}/{len(class_samples)})")
    
    # Hallucinations
    unknown = sum(1 for r in all_results if r['predicted'] == 'Unknown')
    print(f"\n🔍 Hallucinations: {unknown} ({unknown/total*100:.1f}%)")
    
    # Sauvegarder
    final_results = {
        'global_accuracy': global_accuracy,
        'total_samples': total,
        'correct': correct,
        'class_metrics': class_metrics,
        'hallucinations': unknown,
        'version': 'v3_anti_overfitting'
    }
    
    with open(CHECKPOINT_DIR / "test_results_v3.json", 'w') as f:
        json.dump(final_results, f, indent=2)
    
    print(f"\n✅ Résultats sauvegardés: {CHECKPOINT_DIR / 'test_results_v3.json'}")
    
    # Décision finale
    print(f"\n" + "="*60)
    print("DÉCISION FINALE")
    print(f"="*60)
    
    cgm_ok = class_metrics.get('Cassava Green Mottle (CGM)', {}).get('accuracy', 0) >= 30
    cbb_ok = class_metrics.get('Cassava Bacterial Blight (CBB)', {}).get('accuracy', 0) >= 30
    
    if global_accuracy >= 75 and cgm_ok and cbb_ok:
        print("✅✅✅ SUCCÈS COMPLET! Prêt pour Stage 2")
    elif global_accuracy >= 60:
        print("⚠️ Succès partiel - amélioration nécessaire sur CGM/CBB")
        if not cgm_ok:
            print(f"   ❌ CGM < 30%")
        if not cbb_ok:
            print(f"   ❌ CBB < 30%")
    else:
        print("❌ Échec - réentraînement nécessaire")
else:
    print("⏭️ Full evaluation skipped (RUN_FULL_EVAL=False)")


FULL EVALUATION ON TEST SET
⏱️ Temps estimé: ~121 minutes


Full test: 100%|██████████| 3619/3619 [41:21<00:00,  1.46it/s]


RÉSULTATS FINAUX

📊 ACCURACY GLOBALE: 65.05% (2354/3619)

📈 ACCURACY PAR CLASSE:
   ✅ Cassava Bacterial Blight (CBB): 34.0% (53/156)
   ❌ Cassava Brown Streak Disease (CBSD): 2.9% (19/662)
   ✅ Cassava Green Mottle (CGM): 81.4% (258/317)
   ✅ Cassava Mosaic Disease (CMD): 89.4% (1862/2082)
   ✅ Healthy: 40.3% (162/402)

🔍 Hallucinations: 1 (0.0%)

✅ Résultats sauvegardés: /home/hounfodjidagba/learning/cassava/new_implementation/checkpoints/stage1_cassava_gemma3n_v3/test_results_v3.json

DÉCISION FINALE
⚠️ Succès partiel - amélioration nécessaire sur CGM/CBB


## 14. Résumé des Améliorations

In [ ]:
# ============================================================
# CELL 14: RÉSUMÉ DES AMÉLIORATIONS
# ============================================================

print("""
╔══════════════════════════════════════════════════════════════╗
║           RÉSUMÉ DES AMÉLIORATIONS V3                       ║
╠══════════════════════════════════════════════════════════════╣
║                                                              ║
║  🔧 LoRA CONSERVATIF:                                        ║
║     • alpha = r (ratio 1:1 au lieu de 2:1)                   ║
║     • dropout = 0 (optimisé Unsloth)                         ║
║     • Seulement q_proj + v_proj                              ║
║                                                              ║
║  📉 RÉGULARISATION RENFORCÉE:                                ║
║     • weight_decay: 0.01 (était 0.1 = trop!)                 ║
║     • learning_rate: 1e-5 (3x plus lent)                     ║
║     • Early stopping après 5 evals sans amélioration         ║
║                                                              ║
║  ⚖️ GESTION DÉSÉQUILIBRE:                                    ║
║     • Focal Loss avec gamma=2.0                              ║
║     • Class weights: CGM=10, CBB=10, CMD=0.3                 ║
║                                                              ║
║  📝 PROMPT CONTRAINT:                                        ║
║     • Format multiple-choice (A/B/C/D/E)                     ║
║     • Réponses courtes (~30 tokens)                          ║
║     • Élimine les hallucinations                             ║
║                                                              ║
║  🖼️ DATA AUGMENTATION:                                       ║
║     • Géométrique: flip, rotate, crop                        ║
║     • Couleur CONSERVATRICE (préserve maladies)              ║
║     • CLAHE pour visibilité lésions                          ║
║                                                              ║
╚══════════════════════════════════════════════════════════════╝

Si l'overfitting persiste, essayer:
1. Réduire r à 4
2. Augmenter weight_decay à 0.05
3. Réduire learning_rate à 5e-6
4. Augmenter early_stopping_patience à 3
""")

In [19]:
# ============================================================
# DEBUG: Vérifier la distribution des réponses dans le training formaté
# ============================================================

response_distribution = Counter()
for sample in train_formatted[:1000]:  # Vérifier 1000 premiers
    response = sample['conversations'][1]['content']
    response_distribution[response] += 1

print("Distribution des réponses dans le training set:")
for response, count in response_distribution.most_common():
    print(f"  {response}: {count} ({count/10:.1f}%)")

Distribution des réponses dans le training set:
  D) Cassava Mosaic Disease (CMD): 398 (39.8%)
  B) Cassava Brown Streak Disease (CBSD): 194 (19.4%)
  C) Cassava Green Mottle (CGM): 193 (19.3%)
  A) Cassava Bacterial Blight (CBB): 111 (11.1%)
  E) Healthy: 104 (10.4%)


In [20]:
# ============================================================
# DEBUG: Voir les réponses brutes du modèle
# ============================================================

print("\n=== RÉPONSES BRUTES DU MODÈLE ===")
test_samples = random.sample(test_formatted, 10)

for i, sample in enumerate(test_samples):
    pred, raw_response = predict_single(sample, max_new_tokens=50)
    print(f"\n[{i+1}] GT: {sample['class_label']}")
    print(f"    Pred: {pred}")
    print(f"    Raw: '{raw_response}'")


=== RÉPONSES BRUTES DU MODÈLE ===



[1] GT: Cassava Mosaic Disease (CMD)
    Pred: Cassava Mosaic Disease (CMD)
    Raw: ' and full name only.
model
D) Cassava Mosaic Disease (CMD)'

[2] GT: Cassava Brown Streak Disease (CBSD)
    Pred: Cassava Green Mottle (CGM)
    Raw: ' and full name only.
model
C) Cassava Green Mottle (CGM)'

[3] GT: Cassava Mosaic Disease (CMD)
    Pred: Cassava Mosaic Disease (CMD)
    Raw: ' and full name only.
model
D) Cassava Mosaic Disease (CMD)'

[4] GT: Cassava Mosaic Disease (CMD)
    Pred: Cassava Bacterial Blight (CBB)
    Raw: ' and full name only.
model
A) Cassava Bacterial Blight (CBB)'

[5] GT: Cassava Mosaic Disease (CMD)
    Pred: Cassava Mosaic Disease (CMD)
    Raw: ' and full name only.
model
D) Cassava Mosaic Disease (CMD)'

[6] GT: Cassava Green Mottle (CGM)
    Pred: Cassava Green Mottle (CGM)
    Raw: ' and full name only.
model
C) Cassava Green Mottle (CGM)'

[7] GT: Cassava Mosaic Disease (CMD)
    Pred: Cassava Green Mottle (CGM)
    Raw: ' and full name only.
model
C) Ca